<a href="https://colab.research.google.com/github/MorreyB/PC-Free/blob/main/Linux_Virtual_PC_5.0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, subprocess, shutil, psutil, time, threading, datetime, socket
import google.colab
from google.colab import drive, output
from IPython.display import HTML, display

# --- 1. INITIALIZE & MOUNT DRIVE ---
drive.mount('/content/drive', force_remount=True)
P_BASE = '/content/drive/MyDrive/colab_desktop_data'
P_WORKSPACE = os.path.join(P_BASE, 'workspace')
os.makedirs(P_WORKSPACE, exist_ok=True)

# Create a symlink in /content for easy access
if not os.path.exists('/content/workspace'):
    os.symlink(P_WORKSPACE, '/content/workspace')

# --- 2. AUTOMATION: BACKGROUND CLEANUP ---
def auto_cleanup_backups(keep=5):
    backup_root = os.path.join(P_BASE, 'session_backups')
    while True:
        if os.path.exists(backup_root):
            backups = sorted([d for d in os.listdir(backup_root) if os.path.isdir(os.path.join(backup_root, d))])
            if len(backups) > keep:
                for i in range(len(backups) - keep):
                    target = os.path.join(backup_root, backups[i])
                    shutil.rmtree(target)
        time.sleep(3600)

threading.Thread(target=auto_cleanup_backups, daemon=True).start()

# --- 3. SYSTEM DEPENDENCIES & noVNC 1.7.0 ---
print("፤ Installing Dependencies (XFCE, VNC, Audio, noVNC 1.7.0)...")
subprocess.run("add-apt-repository -y ppa:mozillateam/ppa", shell=True, capture_output=True)
subprocess.run("apt-get update", shell=True, capture_output=True)
subprocess.run("apt-get install -y xfce4 xfce4-goodies tightvncserver python3-websockify sudo firefox flatpak libnvidia-gl-535 libegl1 mesa-utils wget pulseaudio pulseaudio-utils libpulse0 pavucontrol rsync", shell=True, capture_output=True)

# Setup noVNC 1.7.0
if not os.path.exists('/opt/novnc'):
    subprocess.run("wget -q https://github.com/novnc/noVNC/archive/v1.7.0.tar.gz -O /tmp/novnc.tar.gz", shell=True)
    subprocess.run("tar -xzf /tmp/novnc.tar.gz -C /opt/", shell=True)
    subprocess.run("mv /opt/noVNC-1.7.0 /opt/novnc", shell=True)
    subprocess.run("cp /opt/novnc/vnc.html /opt/novnc/index.html", shell=True)

# --- 4. APP PERSISTENCE & AUTO-RESTORE ---
def ensure_apps_installed():
    print("፤ Checking persistent apps (Sober, Discord)...")
    flatpak_backup = os.path.join(P_WORKSPACE, 'flatpak_data')

    # If backup exists on Drive, sync it to /var/lib/flatpak
    if os.path.exists(flatpak_backup):
        print("፤ Restoring apps from Drive backup...")
        subprocess.run(f"rsync -a {flatpak_backup}/ /var/lib/flatpak/", shell=True)
        subprocess.run("flatpak repair", shell=True, capture_output=True)

    # Ensure flathub is registered
    subprocess.run("flatpak remote-add --if-not-exists flathub https://dl.flathub.org/repo/flathub.flatpakrepo", shell=True)

    # Final check: Install if still missing
    apps = ["org.vinegarhq.Sober", "com.discordapp.Discord"]
    for app in apps:
        check = subprocess.run(f"flatpak info {app}", shell=True, capture_output=True)
        if check.returncode != 0:
            print(f"፤ Installing {app}...")
            subprocess.run(f"flatpak install flathub {app} -y", shell=True)

ensure_apps_installed()

# --- 5. SESSION MANAGEMENT ---
def save_session():
    ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    bdir = os.path.join(P_BASE, 'session_backups', ts)
    os.makedirs(bdir, exist_ok=True)
    # Backup configs
    for src, name in [('/root/.vnc', 'vnc_config'), ('/root/.config/xfce4', 'xfce4_settings')]:
        if os.path.exists(src): shutil.copytree(src, os.path.join(bdir, name), dirs_exist_ok=True)
    # Backup Flatpak data to Drive
    flatpak_backup = os.path.join(P_WORKSPACE, 'flatpak_data')
    os.makedirs(flatpak_backup, exist_ok=True)
    subprocess.run(f"rsync -a /var/lib/flatpak/ {flatpak_backup}/", shell=True)
    print(f"፤ Session and Apps saved: {ts}")

def restore_session():
    backup_root = os.path.join(P_BASE, 'session_backups')
    if not os.path.exists(backup_root): return
    backups = sorted([d for d in os.listdir(backup_root)])
    if backups:
        target = os.path.join(backup_root, backups[-1])
        print(f"፤ Restoring Settings from: {backups[-1]}")
        for dest, name in [('/root/.vnc', 'vnc_config'), ('/root/.config/xfce4', 'xfce4_settings')]:
            src = os.path.join(target, name)
            if os.path.exists(src):
                if os.path.exists(dest): shutil.rmtree(dest)
                shutil.copytree(src, dest)

# --- 6. START VNC & AUDIO ENGINE ---
subprocess.run("pkill -9 -f Xtightvnc|websockify|dbus|pulseaudio", shell=True)
restore_session()
!rm -rf /tmp/.X* /tmp/.X11-unix /root/.vnc/*.pid /root/.vnc/*.log
os.environ['USER'] = 'root'
os.makedirs("/root/.vnc", exist_ok=True)
if not os.path.exists("/root/.vnc/passwd"):
    subprocess.run("echo password | vncpasswd -f > /root/.vnc/passwd && chmod 600 /root/.vnc/passwd", shell=True)

with open("/root/.vnc/xstartup", "w") as f:
    f.write("#!/bin/sh\nexport DISPLAY=:10\nexport VGL_DISPLAY=egl\ndbus-launch --exit-with-session startxfce4 &\n")
subprocess.run("chmod +x /root/.vnc/xstartup", shell=True)
subprocess.run("vncserver :10 -geometry 1920x1080 -depth 24", shell=True)

# Start Audio
env = os.environ.copy()
env["DISPLAY"] = ":10"
subprocess.Popen(["pulseaudio", "--start", "--exit-idle-time=-1"], env=env)
subprocess.run("pactl load-module module-native-protocol-unix socket=/tmp/pulse-socket", shell=True, env=env)
subprocess.run("xprop -root -f PULSE_SERVER 8s -set PULSE_SERVER /tmp/pulse-socket", shell=True, env=env)

# Start noVNC Proxy
subprocess.Popen("websockify --web /opt/novnc 6080 127.0.0.1:5910 --heartbeat=30", shell=True)

# --- 7. UTILITIES ---
proxy_url = output.eval_js("google.colab.kernel.proxyPort(6080)")
if not proxy_url.endswith('/'): proxy_url += '/'
final_link = f"{proxy_url}vnc.html?autoconnect=true&reconnect=true"

display(HTML(f'<div style="padding:15px; border:2px solid #34a853; border-radius:10px;"><b>Desktop Link:</b> <a href="{final_link}" target="_blank">{final_link}</a></div>'))
print("፤ Initialization complete.")

Mounted at /content/drive
፤ Installing Dependencies (XFCE, VNC, Audio, noVNC 1.7.0)...
፤ Checking persistent apps (Sober, Discord)...
፤ Restoring apps from Drive backup...


In [1]:
import os, subprocess, shutil, psutil, time, threading, torch, datetime, socket
import google.colab
from google.colab import drive, output
from IPython.display import HTML, display

# --- 1. INITIALIZE & MOUNT DRIVE ---
drive.mount('/content/drive', force_remount=True)
P_BASE = '/content/drive/MyDrive/colab_desktop_data'
P_WORKSPACE = os.path.join(P_BASE, 'workspace')
os.makedirs(P_WORKSPACE, exist_ok=True)

# Create a symlink in /content for easy access
if not os.path.exists('/content/workspace'):
    os.symlink(P_WORKSPACE, '/content/workspace')

# --- 2. AUTOMATION: BACKGROUND CLEANUP ---
def auto_cleanup_backups(keep=5):
    backup_root = os.path.join(P_BASE, 'session_backups')
    while True:
        if os.path.exists(backup_root):
            backups = sorted([d for d in os.listdir(backup_root) if os.path.isdir(os.path.join(backup_root, d))])
            if len(backups) > keep:
                for i in range(len(backups) - keep):
                    target = os.path.join(backup_root, backups[i])
                    print(f"፤ [Auto-Cleanup] Removing old backup: {backups[i]}")
                    shutil.rmtree(target)
        time.sleep(3600)

threading.Thread(target=auto_cleanup_backups, daemon=True).start()

# --- 3. SYSTEM DEPENDENCIES & noVNC 1.7.0 ---
print("፤ Installing Dependencies and noVNC 1.7.0...")
subprocess.run("add-apt-repository -y ppa:mozillateam/ppa", shell=True, capture_output=True)
subprocess.run("apt-get update", shell=True, capture_output=True)
subprocess.run("apt-get install -y xfce4 xfce4-goodies tightvncserver python3-websockify sudo firefox flatpak libnvidia-gl-535 libegl1 mesa-utils wget", shell=True, capture_output=True)

# Download and setup noVNC 1.7.0
if not os.path.exists('/opt/noVNC-1.7.0'):
    subprocess.run("wget -q https://github.com/novnc/noVNC/archive/v1.7.0.tar.gz -O /tmp/novnc.tar.gz", shell=True)
    subprocess.run("tar -xzf /tmp/novnc.tar.gz -C /opt/", shell=True)
    subprocess.run("mv /opt/noVNC-1.7.0 /opt/novnc", shell=True)
    subprocess.run("cp /opt/novnc/vnc.html /opt/novnc/index.html", shell=True)

# Add Flathub repository
subprocess.run("flatpak remote-add --if-not-exists flathub https://flathub.org/repo/flathub.flatpakrepo", shell=True)

# --- 4. SESSION MANAGEMENT (ENHANCED) ---
def save_session():
    ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    bdir = os.path.join(P_BASE, 'session_backups', ts)
    os.makedirs(bdir, exist_ok=True)
    for src, name in [('/root/.vnc', 'vnc_config'), ('/root/.config/xfce4', 'xfce4_settings')]:
        if os.path.exists(src): shutil.copytree(src, os.path.join(bdir, name), dirs_exist_ok=True)
    print(f"፤ Session and Desktop Settings saved to Drive: {ts}")

def restore_session():
    backup_root = os.path.join(P_BASE, 'session_backups')
    if not os.path.exists(backup_root): return
    backups = sorted([d for d in os.listdir(backup_root)])
    if backups:
        target = os.path.join(backup_root, backups[-1])
        print(f"፤ Restoring Desktop Settings from: {backups[-1]}")
        for dest, name in [('/root/.vnc', 'vnc_config'), ('/root/.config/xfce4', 'xfce4_settings')]:
            src = os.path.join(target, name)
            if os.path.exists(src):
                if os.path.exists(dest): shutil.rmtree(dest)
                shutil.copytree(src, dest)

# --- 5. START VNC ENGINE ---
subprocess.run("pkill -9 -f Xtightvnc|websockify|dbus", shell=True)
restore_session()
!rm -rf /tmp/.X* /tmp/.X11-unix /root/.vnc/*.pid /root/.vnc/*.log
os.environ['USER'] = 'root'
os.makedirs("/root/.vnc", exist_ok=True)
if not os.path.exists("/root/.vnc/passwd"):
    subprocess.run("echo password | vncpasswd -f > /root/.vnc/passwd && chmod 600 /root/.vnc/passwd", shell=True)
with open("/root/.vnc/xstartup", "w") as f:
    f.write("#!/bin/sh\nexport DISPLAY=:10\nexport VGL_DISPLAY=egl\nxset s off\nxset -dpms\ndbus-launch --exit-with-session startxfce4 &\n")
subprocess.run("chmod +x /root/.vnc/xstartup", shell=True)
subprocess.run("vncserver :10 -geometry 1920x1080 -depth 24", shell=True)
subprocess.Popen("websockify --web /opt/novnc 6080 127.0.0.1:5910", shell=True)

# --- 6. UTILITIES ---
proxy_url = output.eval_js("google.colab.kernel.proxyPort(6080)")
if not proxy_url.endswith('/'): proxy_url += '/'
final_link = f"{proxy_url}vnc.html?autoconnect=true&reconnect=true"
display(HTML(f'<div style="padding:15px; border:2px solid #34a853; border-radius:10px;"><b>Desktop Link (noVNC 1.7.0):</b> <a href="{final_link}" target="_blank">{final_link}</a><br><b>Persistent Folder:</b> /content/workspace</div>'))
print("፤ All components initialized with noVNC 1.7.0.")

Mounted at /content/drive
፤ Installing Dependencies and noVNC 1.7.0...
፤ Restoring Desktop Settings from: 20260625_191738


፤ All components initialized with noVNC 1.7.0.


In [ ]:
%%html
<script src="https://cdn.jsdelivr.net/npm/novnc-audio-plugin@1.0.4/novnc-mediamtx-audio.min.js"></script>

In [ ]:
print("፤ Installing GStreamer and ALSA dependencies...")
!apt-get update
!apt-get install -y gstreamer1.0-tools gstreamer1.0-plugins-base gstreamer1.0-plugins-good gstreamer1.0-plugins-bad gstreamer1.0-plugins-ugly gstreamer1.0-alsa
print("✅ GStreamer installation complete.")

፤ Installing GStreamer and ALSA dependencies...
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/mozillateam/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
R

In [ ]:
from IPython.display import display, HTML

chart_data = """
sequenceDiagram
    NoVNC client->>+x11vnc server: Authenticate
    x11vnc server->>+Tcpulse: Spawn Tcpulse with RFB_CLIENT_IP environment set
    NoVNC client->>+Tcpulse: Connect for audio
"""

display(HTML(f'<div class="mermaid">{chart_data}</div>'))
display(HTML('<script src="https://cdn.jsdelivr.net/npm/mermaid/dist/mermaid.min.js"></script><script>mermaid.initialize({startOnLoad:true});</script>'))

In [ ]:
import zipfile
import os
import subprocess

# 1. Extract the audio plugin resources
zip_path = '/content/NoVNCAudio.zip'
extract_path = os.path.join(P_WORKSPACE, 'novnc_audio_fix')

if os.path.exists(zip_path):
    print(f"፤ Extracting {zip_path} to {extract_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("✅ Extraction complete.")
    !ls -R {extract_path}
else:
    print(f"❌ File {zip_path} not found.")

# 2. Setup environment for audio
print("፤ Setting up audio environment...")
subprocess.run("apt-get install -y pulseaudio pulseaudio-utils libpulse-dev", shell=True)

# 3. Check for setup scripts in the zip
setup_script = os.path.join(extract_path, 'setup.sh')
if os.path.exists(setup_script):
    print("፤ Found setup script, executing...")
    subprocess.run(f"chmod +x {setup_script} && {setup_script}", shell=True)
else:
    print("ℹ️ No automated setup script found in ZIP. Manual configuration may be required based on the extracted files.")

፤ Extracting /content/NoVNCAudio.zip to /content/drive/MyDrive/colab_desktop_data/workspace/novnc_audio_fix...
✅ Extraction complete.
/content/drive/MyDrive/colab_desktop_data/workspace/novnc_audio_fix:
bin  etc  login  README.md  sound  var

/content/drive/MyDrive/colab_desktop_data/workspace/novnc_audio_fix/bin:
update_allowed

/content/drive/MyDrive/colab_desktop_data/workspace/novnc_audio_fix/etc:
allowedhosts  init.d  X11

/content/drive/MyDrive/colab_desktop_data/workspace/novnc_audio_fix/etc/init.d:
vnc

/content/drive/MyDrive/colab_desktop_data/workspace/novnc_audio_fix/etc/X11:
xorg.conf.d

/content/drive/MyDrive/colab_desktop_data/workspace/novnc_audio_fix/etc/X11/xorg.conf.d:
depth.conf

/content/drive/MyDrive/colab_desktop_data/workspace/novnc_audio_fix/login:
main.c	Makefile

/content/drive/MyDrive/colab_desktop_data/workspace/novnc_audio_fix/sound:
Makefile  tcpulse.c

/content/drive/MyDrive/colab_desktop_data/workspace/novnc_audio_fix/var:
www

/content/drive/MyDrive/col

In [ ]:
import os
import subprocess

# Define paths
audio_src_dir = os.path.join(P_WORKSPACE, 'novnc_audio_fix', 'sound')
tcpulse_c_path = os.path.join(audio_src_dir, 'tcpulse.c')

if os.path.exists(tcpulse_c_path):
    print("፤ Patching tcpulse.c syntax error...")
    with open(tcpulse_c_path, 'r') as f:
        content = f.read()

    # Fix the missing semicolon reported in the error log
    # The error was: return ret (missing ;) before }
    fixed_content = content.replace('return ret\n}', 'return ret;\n}')

    with open(tcpulse_c_path, 'w') as f:
        f.write(fixed_content)

print("፤ Re-compiling tcpulse with updated flags...")
# Update Makefile to include necessary libraries and fix linking issues
makefile_content = """TARGETS=tcpulse
CFLAGS += -O2 -fPIC

all: $(TARGETS)

tcpulse: tcpulse.o
	$(CC) $^ -lssl -lcrypto -lpthread -o $@

tcpulse.o: tcpulse.c

clean:
	rm -f tcpulse *.o
"""
with open(os.path.join(audio_src_dir, 'Makefile'), 'w') as f:
    f.write(makefile_content)

# Compile
result = subprocess.run(f"cd {audio_src_dir} && make clean && make", shell=True, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

if result.returncode == 0:
    tcpulse_bin = os.path.join(audio_src_dir, 'tcpulse')
    subprocess.run(f"cp {tcpulse_bin} /usr/local/bin/tcpulse", shell=True)
    print("✅ tcpulse successfully compiled and installed.")

    # Restarting Websockify with the enhanced web assets
    extracted_web_dir = os.path.join(P_WORKSPACE, 'novnc_audio_fix', 'var/www/vnc')
    subprocess.run("pkill -9 -f websockify", shell=True)
    subprocess.Popen(f"websockify --web {extracted_web_dir} 6080 127.0.0.1:5910 --heartbeat=30", shell=True)
    print("✅ Websockify restarted with audio-enabled assets.")
else:
    print("❌ Compilation failed again. Checking specific errors.")

፤ Patching tcpulse.c syntax error...
፤ Re-compiling tcpulse with updated flags...
rm -f tcpulse *.o
cc -O2 -fPIC   -c -o tcpulse.o tcpulse.c
cc tcpulse.o -lssl -lcrypto -lpthread -o tcpulse

tcpulse.c: In function ‘hash_key’:
tcpulse.c:142:5: warning: ‘SHA1_Init’ is deprecated: Since OpenSSL 3.0 [-Wdeprecated-declarations]
  142 |     SHA1_Init(&c);
      |     ^~~~~~~~~
In file included from /usr/include/openssl/x509.h:41,
                 from /usr/include/openssl/ssl.h:31,
                 from tcpulse.c:26:
/usr/include/openssl/sha.h:49:27: note: declared here
   49 | OSSL_DEPRECATEDIN_3_0 int SHA1_Init(SHA_CTX *c);
      |                           ^~~~~~~~~
tcpulse.c:143:5: warning: ‘SHA1_Update’ is deprecated: Since OpenSSL 3.0 [-Wdeprecated-declarations]
  143 |     SHA1_Update(&c, key, strlen(key));
      |     ^~~~~~~~~~~
In file included from /usr/include/openssl/x509.h:41,
                 from /usr/include/openssl/ssl.h:31,
                 from tcpulse.c:26:
/usr/include/

In [ ]:
import os

# Listing the contents of the extraction directory to look for config files or READMEs
fix_path = os.path.join(P_WORKSPACE, 'novnc_audio_fix')
print(f"፤ Auditing contents of {fix_path}:\n")

for root, dirs, files in os.walk(fix_path):
    level = root.replace(fix_path, '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 4 * (level + 1)
    for f in files:
        print(f"{subindent}{f}")

፤ Auditing contents of /content/drive/MyDrive/colab_desktop_data/workspace/novnc_audio_fix:



In [ ]:
# Read the README if it exists to find specific setup instructions
readme_path = os.path.join(P_WORKSPACE, 'novnc_audio_fix', 'README.md')
if os.path.exists(readme_path):
    print("\n--- README.md Content ---")
    with open(readme_path, 'r') as f:
        print(f.read())
else:
    print("\nℹႀ No README.md found in the audio fix package.")


ℹႀ No README.md found in the audio fix package.


In [ ]:
import subprocess
import os
import time

print("፤ Installing audio dependencies...")
subprocess.run("apt-get install -y pulseaudio libpulse0 pavucontrol pulseaudio-utils", shell=True, capture_output=True)

print("፤ Resetting Audio System...")
# Kill old processes
subprocess.run("pkill -9 pulseaudio", shell=True, capture_output=True)
subprocess.run("pkill -9 websockify", shell=True, capture_output=True)
subprocess.run("rm -rf /root/.config/pulse /tmp/pulse-* /tmp/pulse-socket", shell=True, capture_output=True)

# Start Pulseaudio
env = os.environ.copy()
env["DISPLAY"] = ":10"
subprocess.Popen(["pulseaudio", "--start", "--exit-idle-time=-1"], env=env)
time.sleep(2)

# Load modules for noVNC bridge
print("፤ Configuring virtual audio sink and unix socket...")
subprocess.run("pactl load-module module-null-sink sink_name=vnc_sink", shell=True, env=env)
subprocess.run("pactl set-default-sink vnc_sink", shell=True, env=env)
subprocess.run("pactl load-module module-native-protocol-unix socket=/tmp/pulse-socket", shell=True, env=env)

# Map to X11 so noVNC can find it
subprocess.run("xprop -root -f PULSE_SERVER 8s -set PULSE_SERVER /tmp/pulse-socket", shell=True, env=env)

# Restart Gateway with Audio support
print("፤ Restarting noVNC Gateway with Audio Bridge...")
subprocess.Popen("websockify --web /opt/novnc 6080 127.0.0.1:5910 --heartbeat=30", shell=True)

print("\n✅ Audio Bridge is Active.")
print("👉 1. Refresh your noVNC tab.")
print("👉 2. In the VNC sidebar, go to Settings > Local Speaker and turn it ON.")

፤ Installing audio dependencies...
፤ Resetting Audio System...
፤ Configuring virtual audio sink and unix socket...
፤ Restarting noVNC Gateway with Audio Bridge...

✅ Audio Bridge is Active.
👉 1. Refresh your noVNC tab.
👉 2. In the VNC sidebar, go to Settings > Local Speaker and turn it ON.


In [ ]:
import torch

def check_vram():
    if torch.cuda.is_available():
        device = torch.cuda.current_device()
        total_memory = torch.cuda.get_device_properties(device).total_memory / (1024**3)
        allocated_memory = torch.cuda.memory_allocated(device) / (1024**3)
        cached_memory = torch.cuda.memory_reserved(device) / (1024**3)
        print(f'፤ GPU: {torch.cuda.get_device_name(device)}')
        print(f'፤ Total VRAM: {total_memory:.2f} GB')
        print(f'፤ Allocated VRAM: {allocated_memory:.2f} GB')
        print(f'፤ Cached VRAM: {cached_memory:.2f} GB')
    else:
        print('፤ No GPU detected. Please ensure the runtime type is set to GPU (Edit > Notebook settings).')

def clear_vram():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print('፤ GPU Cache cleared to optimize performance.')

check_vram()

፤ GPU: Tesla T4
፤ Total VRAM: 14.56 GB
፤ Allocated VRAM: 0.00 GB
፤ Cached VRAM: 0.00 GB


In [ ]:
import subprocess
# Display the contents of /etc/os-release to identify the Linux distribution
result = subprocess.run(['cat', '/etc/os-release'], capture_output=True, text=True)
print(result.stdout)

PRETTY_NAME="Ubuntu 22.04.5 LTS"
NAME="Ubuntu"
VERSION_ID="22.04"
VERSION="22.04.5 LTS (Jammy Jellyfish)"
VERSION_CODENAME=jammy
ID=ubuntu
ID_LIKE=debian
HOME_URL="https://www.ubuntu.com/"
SUPPORT_URL="https://help.ubuntu.com/"
BUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"
PRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"
UBUNTU_CODENAME=jammy



### 🚚 Move Stray Files to Permanent Workspace
If you accidentally saved files in the default `/content` folder, use this cell to move them into your persistent Google Drive workspace.

In [ ]:
import os, shutil, requests

def setup_persistent_ref():
    ref_url = 'https://raw.githubusercontent.com/vinegarhq/sober/main/flatpak/org.vinegarhq.Sober.flatpakref'
    target_path = os.path.join(P_WORKSPACE, 'org.vinegarhq.Sober.flatpakref')

    print(f"፤ Downloading Sober reference to persistent workspace...")
    r = requests.get(ref_url)
    with open(target_path, 'wb') as f:
        f.write(r.content)

    if os.path.exists(target_path):
        print(f"፤ Success: File is now safe in Drive at: {target_path}")
    else:
        print("፤ Error: Failed to save file to Drive.")

setup_persistent_ref()

፤ Downloading Sober reference to persistent workspace...
፤ Success: File is now safe in Drive at: /content/drive/MyDrive/colab_desktop_data/workspace/org.vinegarhq.Sober.flatpakref


### 🔍 Drive Workspace Verification
This cell confirms that your workspace is correctly linked to your Google Drive for permanent storage.

In [ ]:
import os

# Check if the symlink exists and points to Drive
if os.path.islink('/content/workspace'):
    target = os.readlink('/content/workspace')
    print(f"✅ Workspace Link Active: /content/workspace -> {target}")
else:
    print("❌ Workspace link missing. Re-running the Master Cell (9dc8fa53) will fix this.")

# List files currently in the permanent Drive folder
print("\n📂 Files currently safe on Drive:")
!ls -lh /content/workspace

✅ Workspace Link Active: /content/workspace -> /content/drive/MyDrive/colab_desktop_data/workspace

📂 Files currently safe on Drive:
lrwxrwxrwx 1 root root 51 Jun 25 15:44 /content/workspace -> /content/drive/MyDrive/colab_desktop_data/workspace


In [ ]:
import shutil
import os

source_file = '/content/org.vinegarhq.Sober.flatpakref'
destination_folder = '/content/workspace'

if os.path.exists(source_file):
    shutil.move(source_file, os.path.join(destination_folder, os.path.basename(source_file)))
    print(f"✅ Successfully moved {source_file} to persistent storage.")
else:
    print("ℹ️ File already moved or not found in temporary storage.")

# Final check of the workspace
print("\n📂 Updated persistent files:")
!ls -lh /content/workspace

ℹ️ File already moved or not found in temporary storage.

📂 Updated persistent files:
lrwxrwxrwx 1 root root 51 Jun 25 15:44 /content/workspace -> /content/drive/MyDrive/colab_desktop_data/workspace


In [ ]:
import shutil
import os

# List of stray files in /content that should be saved
stray_files = [
    '/content/NoVNCAudio.zip',
    '/content/geckodriver-v0.34.0-linux64.tar.gz',
    '/content/google-chrome-stable_current_amd64.deb'
]

dest_dir = '/content/workspace'

print("፤ Moving stray files to permanent workspace...")
for file_path in stray_files:
    if os.path.exists(file_path):
        file_name = os.path.basename(file_path)
        try:
            shutil.move(file_path, os.path.join(dest_dir, file_name))
            print(f"✅ Moved {file_name} to {dest_dir}")
        except Exception as e:
            print(f"❌ Could not move {file_name}: {e}")
    else:
        print(f"ℹ️ {os.path.basename(file_path)} already moved or not found.")

print("\n📂 Final Workspace Audit:")
!ls -lh /content/workspace

፤ Moving stray files to permanent workspace...
✅ Moved NoVNCAudio.zip to /content/workspace
✅ Moved geckodriver-v0.34.0-linux64.tar.gz to /content/workspace
✅ Moved google-chrome-stable_current_amd64.deb to /content/workspace

📂 Final Workspace Audit:
lrwxrwxrwx 1 root root 51 Jun 26 04:56 /content/workspace -> /content/drive/MyDrive/colab_desktop_data/workspace


In [ ]:
import subprocess
import os
import time

print("፤ Hard-resetting PulseAudio for testing...")
# 1. Kill and cleanup
subprocess.run("pkill -9 pulseaudio", shell=True, capture_output=True)
subprocess.run("rm -rf /tmp/pulse-* /tmp/pulse-socket", shell=True, capture_output=True)

# 2. Start PulseAudio with explicit configuration
env = os.environ.copy()
env["DISPLAY"] = ":10"
# Start daemon
subprocess.Popen(["pulseaudio", "--start", "--exit-idle-time=-1", "--realtime=false"], env=env)
time.sleep(3)

# 3. Load the specific bridge module needed for noVNC audio
print("፤ Loading native protocol unix socket bridge...")
subprocess.run("pactl load-module module-native-protocol-unix socket=/tmp/pulse-socket", shell=True, env=env)

# 4. Verify and Test
sound_file = "/usr/share/sounds/freedesktop/stereo/complete.oga"
if os.path.exists(sound_file):
    print(f"፤ Testing playback with explicit server path...")
    # Point directly to the socket we just created
    result = subprocess.run(["paplay", "--server=unix:/tmp/pulse-socket", sound_file], env=env, capture_output=True, text=True)
    if result.returncode == 0:
        print("✅ Success! Audio signal sent to bridge.")
        print("👉 Check your noVNC tab now (ensure Local Speaker is ON).")
    else:
        print(f"❌ Audio still failing: {result.stderr}")
else:
    print("ℹ️ System sound file not found, check the VNC volume mixer manually.")

፤ Hard-resetting PulseAudio for testing...
፤ Loading native protocol unix socket bridge...
፤ Testing playback with explicit server path...
✅ Success! Audio signal sent to bridge.
👉 Check your noVNC tab now (ensure Local Speaker is ON).


In [ ]:
import subprocess
import os
import time

print("፤ Resolving 500 Error & Restarting Gateway...")

# 1. Define paths
audio_web_dir = os.path.join(P_WORKSPACE, 'novnc_audio_fix/var/www/vnc')

# 2. Fix index.html (Websockify/Browser expectation)
if os.path.exists(audio_web_dir):
    index_path = os.path.join(audio_web_dir, 'index.html')
    vnc_html = os.path.join(audio_web_dir, 'vnc.html')

    # Remove if it exists as a broken link, then recreate
    if os.path.islink(index_path) or os.path.exists(index_path):
        os.remove(index_path)

    if os.path.exists(vnc_html):
        print(f"፤ Linking {vnc_html} to index.html...")
        os.symlink('vnc.html', index_path)

# 3. Clean and Restart
subprocess.run("pkill -9 -f websockify", shell=True)
time.sleep(2)

# Start websockify with absolute path
print(f"፤ Serving assets from: {audio_web_dir}")
subprocess.Popen(f"websockify --web {audio_web_dir} 6080 127.0.0.1:5910 --heartbeat=30", shell=True)

time.sleep(3)
print("\n✅ Gateway restart command sent. Checking status...")
!ps aux | grep websockify

፤ Resolving 500 Error & Restarting Gateway...
፤ Serving assets from: /content/drive/MyDrive/colab_desktop_data/workspace/novnc_audio_fix/var/www/vnc

✅ Gateway restart command sent. Checking status...
root       40586  0.0  0.0   2892  1816 ?        S    05:38   0:00 /bin/sh -c websockify --web /content/drive/MyDrive/colab_desktop_data/workspace/novnc_audio_fix/var/www/vnc 6080 127.0.0.1:5910 --heartbeat=30
root       40587 11.0  0.2 134904 37092 ?        Sl   05:38   0:00 /usr/bin/python3 /usr/bin/websockify --web /content/drive/MyDrive/colab_desktop_data/workspace/novnc_audio_fix/var/www/vnc 6080 127.0.0.1:5910 --heartbeat=30
root       40601  0.0  0.0   7372  3584 ?        S    05:39   0:00 /bin/bash -c ps aux | grep websockify
root       40603  0.0  0.0   6480  2408 ?        S    05:39   0:00 grep websockify


In [ ]:
import requests
import os
import subprocess
import time
import shutil

# 1. Path Management
web_dir = os.path.realpath(os.path.join(P_WORKSPACE, 'novnc_audio_fix/var/www/vnc'))
index_file = os.path.join(web_dir, 'index.html')
vnc_source = os.path.join(web_dir, 'vnc.html')

print(f"፤ Web Root: {web_dir}")

if os.path.exists(web_dir):
    # Use a physical copy instead of a symlink to avoid 404/permission issues
    if os.path.islink(index_file) or os.path.exists(index_file):
        os.remove(index_file)

    if os.path.exists(vnc_source):
        shutil.copy2(vnc_source, index_file)
        print(f"✅ Created physical copy: {index_file}")

    # 2. Restart Gateway with absolute path and explicit directory check
    subprocess.run("pkill -9 -f websockify", shell=True)
    time.sleep(2)

    print("፤ Starting Websockify...")
    # Explicitly change to the directory to avoid any relative path resolution errors
    subprocess.Popen(f"cd {web_dir} && websockify --web . 6080 127.0.0.1:5910 --heartbeat=30", shell=True)
    time.sleep(5)

    # 3. Final Verification
    test_url = 'http://127.0.0.1:6080/'
    try:
        response = requests.get(test_url, timeout=5)
        print(f"📡 Probe (/) Status: {response.status_code}")
        if response.status_code == 200:
            print("✅ SUCCESS: The 500/404 errors are resolved. Refresh your VNC tab!")
        else:
            print(f"❌ Server returned: {response.status_code}")
    except Exception as e:
        print(f"❌ Probe failed: {e}")
else:
    print("❌ Error: Web directory not found.")

፤ Web Root: /content/drive/MyDrive/colab_desktop_data/workspace/novnc_audio_fix/var/www/vnc
፤ Starting Websockify...
📡 Probe (/) Status: 200
✅ SUCCESS: The 500/404 errors are resolved. Refresh your VNC tab!


In [ ]:
import shutil
import os
import subprocess
import time

# 1. Define paths
drive_web_dir = os.path.join(P_WORKSPACE, 'novnc_audio_fix/var/www/vnc')
local_web_dir = '/tmp/vnc_web'

print(f"፤ Migrating web assets from Drive to local storage...")

# 2. Copy files locally to ensure high-performance access and avoid 500 errors
if os.path.exists(local_web_dir): shutil.rmtree(local_web_dir)
shutil.copytree(drive_web_dir, local_web_dir)

# 3. Ensure index.html exists physically
local_index = os.path.join(local_web_dir, 'index.html')
local_vnc = os.path.join(local_web_dir, 'vnc.html')
if not os.path.exists(local_index) and os.path.exists(local_vnc):
    shutil.copy2(local_vnc, local_index)

# 4. Restart Websockify from the local path
subprocess.run("pkill -9 -f websockify", shell=True)
time.sleep(2)

print(f"፤ Starting Websockify from local directory: {local_web_dir}")
subprocess.Popen(f"websockify --web {local_web_dir} 6080 127.0.0.1:5910 --heartbeat=30", shell=True)
time.sleep(5)

# 5. Local Probe
import requests
try:
    res = requests.get('http://127.0.0.1:6080/', timeout=5)
    print(f"📡 Local Probe Status: {res.status_code}")
    if res.status_code == 200:
        print("✅ SUCCESS: Web server is responding locally. Refresh your VNC tab now!")
except Exception as e:
    print(f"❌ Probe failed: {e}")

፤ Migrating web assets from Drive to local storage...
፤ Starting Websockify from local directory: /tmp/vnc_web
📡 Local Probe Status: 200
✅ SUCCESS: Web server is responding locally. Refresh your VNC tab now!


In [ ]:
import shutil
import os
import subprocess
import time

# 1. Define paths
drive_web_dir = os.path.join(P_WORKSPACE, 'novnc_audio_fix/var/www/vnc')
local_web_dir = '/tmp/vnc_web'

print(f"፤ Migrating web assets to local disk for reliability...")

# 2. Copy files locally
if os.path.exists(local_web_dir): shutil.rmtree(local_web_dir)
if os.path.exists(drive_web_dir):
    shutil.copytree(drive_web_dir, local_web_dir)

# 3. Ensure entry point (index.html) exists locally
local_index = os.path.join(local_web_dir, 'index.html')
source_vnc = os.path.join(drive_web_dir, 'vnc.html')

if not os.path.exists(local_index) and os.path.exists(source_vnc):
    shutil.copy2(source_vnc, local_index)

# 4. Restart Websockify from the local path
subprocess.run("pkill -9 -f websockify", shell=True)
time.sleep(2)

print(f"፤ Restarting Gateway from: {local_web_dir}")
subprocess.Popen(f"websockify --web {local_web_dir} 6080 127.0.0.1:5910 --heartbeat=30", shell=True)
time.sleep(5)

# 5. Local Probe
import requests
try:
    res = requests.get('http://127.0.0.1:6080/', timeout=5)
    print(f"📡 Local Health Check: {res.status_code}")
    if res.status_code == 200:
        print("✅ SUCCESS: Server is ready locally. Please refresh your browser tab!")
except Exception as e:
    print(f"❌ Probe failed: {e}")

፤ Migrating web assets to local disk for reliability...
፤ Restarting Gateway from: /tmp/vnc_web
📡 Local Health Check: 200
✅ SUCCESS: Server is ready locally. Please refresh your browser tab!


In [ ]:
import subprocess
import os
import shutil

def ensure_virtualgl():
    if not os.path.exists('/opt/VirtualGL/bin/vglrun'):
        print('፤ VirtualGL not found. Installing...')
        # Download and install VirtualGL 3.1
        subprocess.run('wget https://sourceforge.net/projects/virtualgl/files/3.1/virtualgl_3.1_amd64.deb/download -O vgl.deb', shell=True, capture_output=True)
        subprocess.run('dpkg -i vgl.deb && apt-get install -f -y', shell=True, capture_output=True)
        os.remove('vgl.deb')
        print('፤ VirtualGL installation complete.')

def launch_sober_persistent():
    ensure_virtualgl()

    env = os.environ.copy()
    env.update({
        "DISPLAY": ":10",
        "VGL_DISPLAY": "egl",
        "__GLX_VENDOR_LIBRARY_NAME": "nvidia"
    })

    print("፤ Launching Sober (Roblox) on the virtual desktop...")
    # Path to vglrun
    vgl_path = "/opt/VirtualGL/bin/vglrun"
    subprocess.Popen([vgl_path, "-d", "egl", "flatpak", "run", "org.vinegarhq.Sober"], env=env)

launch_sober_persistent()

፤ VirtualGL not found. Installing...
፤ VirtualGL installation complete.
፤ Launching Sober (Roblox) on the virtual desktop...


In [ ]:
import os
import shutil

# 1. Ensure the Desktop directory exists
desktop_path = '/root/Desktop'
os.makedirs(desktop_path, exist_ok=True)

# 2. Copy the flatpakref to the desktop for easy access
source_ref = '/content/workspace/org.vinegarhq.Sober.flatpakref'
dest_ref = os.path.join(desktop_path, 'org.vinegarhq.Sober.flatpakref')

if os.path.exists(source_ref):
    shutil.copy2(source_ref, dest_ref)
    print(f"✅ Copied Sober reference to desktop: {dest_ref}")

# 3. Create a functional Desktop entry with DBus session support
# Using 'dbus-run-session' ensures the Flatpak can connect to a bus instance
shortcut_content = """[Desktop Entry]
Version=1.0
Type=Application
Name=Sober (Roblox)
Comment=Run Sober via VirtualGL with DBus
Exec=dbus-run-session -- /opt/VirtualGL/bin/vglrun -d egl flatpak run org.vinegarhq.Sober
Icon=org.vinegarhq.Sober
Terminal=false
Categories=Game;
"""

with open(os.path.join(desktop_path, 'Sober.desktop'), 'w') as f:
    f.write(shortcut_content)

# Set execution permissions
os.chmod(os.path.join(desktop_path, 'Sober.desktop'), 0o755)
print("✅ Created updated Sober launch shortcut (with DBus support) on the desktop.")

✅ Copied Sober reference to desktop: /root/Desktop/org.vinegarhq.Sober.flatpakref
✅ Created updated Sober launch shortcut (with DBus support) on the desktop.


In [ ]:
import subprocess

# --- CONFIGURATION ---
# Replace this with any Flathub ID (e.g., 'org.gimp.GIMP', 'com.discordapp.Discord')
APP_ID = 'org.vinegarhq.Sober'

print(f"፤ Ensuring Flathub is configured...")
subprocess.run('flatpak remote-add --if-not-exists flathub https://dl.flathub.org/repo/flathub.flatpakrepo', shell=True)

print(f"፤ Installing {APP_ID}...")
# The -y flag auto-confirms installation
result = subprocess.run(f'flatpak install flathub {APP_ID} -y', shell=True, capture_output=True, text=True)

if result.returncode == 0:
    print(f"✅ {APP_ID} has been successfully installed.")
    print("\nApp Info:")
    subprocess.run(f'flatpak info {APP_ID}', shell=True)
else:
    print(f"❌ Installation failed for {APP_ID}.")
    print("Error Log:", result.stderr)

፤ Ensuring Flathub is configured...
፤ Installing org.vinegarhq.Sober...
✅ org.vinegarhq.Sober has been successfully installed.

App Info:


In [ ]:
import subprocess

# To install a different app, change the ID below:
NEW_APP_ID = 'com.discordapp.Discord'

print(f"፤ Installing {NEW_APP_ID} from Flathub...")
process = subprocess.run(f'flatpak install flathub {NEW_APP_ID} -y', shell=True, capture_output=True, text=True)

if process.returncode == 0:
    print(f"✅ {NEW_APP_ID} installed successfully!")
else:
    print(f"❌ Failed to install {NEW_APP_ID}.")
    print(process.stderr)

፤ Installing com.discordapp.Discord from Flathub...
✅ com.discordapp.Discord installed successfully!


In [ ]:
import os
import subprocess

desktop_shortcut = '/root/Desktop/Sober.desktop'

print("--- Sober Shortcut Verification ---")
# 1. Check if the file exists
if os.path.exists(desktop_shortcut):
    print(f"✅ Shortcut File Found: {desktop_shortcut}")

    # 2. Check if the file is executable
    if os.access(desktop_shortcut, os.X_OK):
        print("✅ Shortcut is Executable")
    else:
        print("❌ Shortcut is NOT executable. Fixing now...")
        os.chmod(desktop_shortcut, 0o755)
else:
    print("❌ Shortcut File Missing!")

# 3. Verify Flatpak registration
print("\n--- Application Registration ---")
flatpak_check = subprocess.run(["flatpak", "info", "org.vinegarhq.Sober"], capture_output=True, text=True)
if flatpak_check.returncode == 0:
    print("✅ Sober is correctly installed via Flatpak.")
else:
    print("❌ Sober Flatpak not found. You may need to run the installation cell again.")

# 4. Instructions for the user
print("\n💡 To verify visually: Open your VNC Desktop Link and look for the 'Sober (Roblox)' icon. Double-click it to start working.")

--- Sober Shortcut Verification ---
✅ Shortcut File Found: /root/Desktop/Sober.desktop
✅ Shortcut is Executable

--- Application Registration ---
✅ Sober is correctly installed via Flatpak.

💡 To verify visually: Open your VNC Desktop Link and look for the 'Sober (Roblox)' icon. Double-click it to start working.


In [ ]:
import subprocess

print("፤ Ensuring Flathub remote is active...")
subprocess.run("flatpak remote-add --if-not-exists flathub https://dl.flathub.org/repo/flathub.flatpakrepo", shell=True)

print("፤ Installing Sober (Roblox) from Flathub...")
# Direct installation from the Flathub ID is more reliable than the ref file in some environments
subprocess.run("flatpak install flathub org.vinegarhq.Sober -y", shell=True)

# Final registration check
print("\n--- Registration Check ---")
!flatpak info org.vinegarhq.Sober

፤ Ensuring Flathub remote is active...
፤ Installing Sober (Roblox) from Flathub...

--- Registration Check ---

VinegarHQ & Sober contributors - Play, chat & explore on Roblox

          ID: org.vinegarhq.Sober
         Ref: app/org.vinegarhq.Sober/x86_64/stable
        Arch: x86_64
      Branch: stable
     Version: 1.7.1
     License: LicenseRef-proprietary=https://sober.vinegarhq.org/notice.txt
      Origin: flathub
  Collection: org.flathub.Stable
Installation: system
   Installed: 17.8 MB
     Runtime: org.gnome.Platform/x86_64/50
         Sdk: org.gnome.Sdk/x86_64/50

      Commit: 2dc01ea5b3a80dedebc0b06d7e674b15d8ee3f01fe35115d78d38afd34882327
      Parent: 102964bde9e5c3f14288b75e0ff54777ae06fdea98be073d679b74fa3d9eccb8
     Subject: Update Sober to 2026-06-22_e24358e (1.7.1) (aa724b7e5301)
        Date: 2026-06-22 19:20:03 +0000


In [ ]:
import subprocess

# 1. Ensure Flathub is added correctly
print('፤ Adding Flathub repository...')
subprocess.run('flatpak remote-add --if-not-exists flathub https://dl.flathub.org/repo/flathub.flatpakrepo', shell=True)

# 2. Update Flatpak remotes
print('፤ Updating Flatpak metadata...')
subprocess.run('flatpak update --appstream', shell=True)

# 3. Install Sober via Flathub ID
print('፤ Installing Sober (Roblox) from Flathub...')
# The -y flag automates the installation process
subprocess.run('flatpak install flathub org.vinegarhq.Sober -y', shell=True)

# 4. Final verification
print('\n--- Installation Verification ---')
check = subprocess.run(['flatpak', 'info', 'org.vinegarhq.Sober'], capture_output=True, text=True)
if check.returncode == 0:
    print('✅ Sober successfully installed and registered via Flathub.')
else:
    print('❌ Installation failed. Please check the logs above.')

፤ Adding Flathub repository...
፤ Updating Flatpak metadata...
፤ Installing Sober (Roblox) from Flathub...

--- Installation Verification ---
✅ Sober successfully installed and registered via Flathub.


In [ ]:
import os
import subprocess

# 1. Force the desktop manager to accept the launcher
shortcut_path = '/root/Desktop/Sober.desktop'
os.chmod(shortcut_path, 0o775)
subprocess.run(['gio', 'set', '-t', 'string', shortcut_path, 'metadata::xfce-exe-checksum', subprocess.getoutput(f'sha256sum {shortcut_path}').split()[0]], check=False)

# 2. Provide the direct terminal launch command for the user
print("--- VNC TERMINAL COMMAND ---")
print("If you are inside the desktop, open the Terminal and run this exact line:")
print("\n/opt/VirtualGL/bin/vglrun -d egl flatpak run org.vinegarhq.Sober\n")

# 3. Attempt to force-launch the app from the kernel into the display
print("፤ Attempting to force-open Sober on Display :10...")
env = os.environ.copy()
env.update({"DISPLAY": ":10", "VGL_DISPLAY": "egl", "__GLX_VENDOR_LIBRARY_NAME": "nvidia"})
subprocess.Popen(["/opt/VirtualGL/bin/vglrun", "-d", "egl", "flatpak", "run", "org.vinegarhq.Sober"], env=env)

print("\n✅ Check your VNC window; the game window should be initializing now.")

--- VNC TERMINAL COMMAND ---
If you are inside the desktop, open the Terminal and run this exact line:

/opt/VirtualGL/bin/vglrun -d egl flatpak run org.vinegarhq.Sober

፤ Attempting to force-open Sober on Display :10...

✅ Check your VNC window; the game window should be initializing now.


In [ ]:
import subprocess
import os

print("--- Fixing DBus and Relaunching Sober ---")

# 1. Ensure a DBus session is running for the current environment
dbus_cmd = "dbus-launch --sh-syntax"
result = subprocess.check_output(dbus_cmd, shell=True, text=True)
for line in result.splitlines():
    if '=' in line:
        name, value = line.split(';', 1)[0].split('=', 1)
        os.environ[name] = value

# 2. Setup the environment for GPU + Display
env = os.environ.copy()
env.update({
    "DISPLAY": ":10",
    "VGL_DISPLAY": "egl",
    "__GLX_VENDOR_LIBRARY_NAME": "nvidia",
    "DBUS_SESSION_BUS_ADDRESS": os.environ.get("DBUS_SESSION_BUS_ADDRESS", "")
})

print(f"፤ Using DBus Address: {env['DBUS_SESSION_BUS_ADDRESS']}")
print("፤ Attempting fresh launch...")

try:
    # Launching with VirtualGL via Flatpak
    subprocess.Popen(
        ["/opt/VirtualGL/bin/vglrun", "-d", "egl", "flatpak", "run", "org.vinegarhq.Sober"],
        env=env
    )
    print("✅ Launch command sent. Please check your VNC Desktop window.")
except Exception as e:
    print(f"❌ Final Launch Error: {e}")

--- Fixing DBus and Relaunching Sober ---
፤ Using DBus Address: 'unix:abstract=/tmp/dbus-lBZSt5oxlT,guid=fba6309e1ea7134700ab84496a3d5531'
፤ Attempting fresh launch...
✅ Launch command sent. Please check your VNC Desktop window.


### 🛠️ Session & App Management
Use this cell to monitor your running applications, clear GPU cache, or force-save your desktop settings to Google Drive.

In [ ]:
import psutil

def manage_environment():
    print("--- 📊 System Status ---")
    check_vram()

    print("\n--- 🎮 Running Apps (Flatpak) ---")
    found_any = False
    for proc in psutil.process_iter(['pid', 'name', 'cmdline']):
        cmd = ' '.join(proc.info['cmdline'] or [])
        if 'flatpak' in cmd and ('Sober' in cmd or 'Discord' in cmd):
            print(f" ✅ {proc.info['name']} (PID: {proc.info['pid']}) is active.")
            found_any = True

    if not found_any:
        print(" ℹ️ No Flatpak games/apps detected as running.")

    print("\n--- 💾 Persistence ---")
    save_session()

# Run Management Summary
manage_environment()

--- 📊 System Status ---
፤ GPU: Tesla T4
፤ Total VRAM: 14.56 GB
፤ Allocated VRAM: 0.00 GB
፤ Cached VRAM: 0.00 GB

--- 🎮 Running Apps (Flatpak) ---
 ℹ️ No Flatpak games/apps detected as running.

--- 💾 Persistence ---
፤ Session and Desktop Settings saved to Drive: 20260625_164858
፤ Note: All files in /content/workspace are already safe on Drive.


### 🚀 Force Launch Sober
Run this cell if the desktop icon is not opening the game. This command manually bridges the GPU to the display session.

In [ ]:
import subprocess
import os

# Ensure environment variables are set for the headless display
env = os.environ.copy()
env.update({
    "DISPLAY": ":10",
    "VGL_DISPLAY": "egl",
    "__GLX_VENDOR_LIBRARY_NAME": "nvidia",
    "DBUS_SESSION_BUS_ADDRESS": os.environ.get("DBUS_SESSION_BUS_ADDRESS", "")
})

print("፤ Attempting to force-launch Sober on Display :10...")
# Use dbus-run-session to ensure communication channels are open
subprocess.Popen(["dbus-run-session", "--", "/opt/VirtualGL/bin/vglrun", "-d", "egl", "flatpak", "run", "org.vinegarhq.Sober"], env=env)

print("✅ Command sent. Switch to your VNC tab to see the window.")

፤ Attempting to force-launch Sober on Display :10...
✅ Command sent. Switch to your VNC tab to see the window.


### 🎧 Audio & Network Diagnostics
Run this cell if you experience no sound or login errors within Sober. It ensures the Pulseaudio and Network namespaces are accessible to your Flatpak apps.

In [ ]:
!bash

bash: cannot set terminal process group (13939): Inappropriate ioctl for device
bash: no job control in this shell
/content# flatpak override --user --share=network org.vinegarhq.Sober


/content# ^C


In [ ]:
import subprocess

def robust_network_fix():
    print("--- 🛠️ Robust Flatpak Network Bridge ---")

    # 1. Apply overrides at all levels to bypass environment restrictions
    apps = ["org.vinegarhq.Sober", "com.discordapp.Discord"]
    for app in apps:
        subprocess.run(["flatpak", "override", "--user", "--share=network", "--share=ipc", app], capture_output=True)
        subprocess.run(["flatpak", "override", "--system", "--share=network", "--share=ipc", app], capture_output=True)

    print("፤ Permissions updated for Sober and Discord.")

    print("\n--- 🔍 Network Namespace Verification ---")
    # Test if the container can see the host's DNS configuration
    check = subprocess.run(["flatpak", "run", "--command=cat", "org.vinegarhq.Sober", "/etc/resolv.conf"], capture_output=True, text=True)

    if "nameserver" in check.stdout:
        print("✅ Success: Network configuration is now visible inside the game container.")
    else:
        print("❌ Restricted: The container is still isolated. Please restart the 'Master Cell' (9dc8fa53) to refresh the XFCE/DBus bridge.")

robust_network_fix()

--- 🛠️ Robust Flatpak Network Bridge ---
፤ Permissions updated for Sober and Discord.

--- 🔍 Network Namespace Verification ---
❌ Restricted: The container is still isolated. Please restart the 'Master Cell' (9dc8fa53) to refresh the XFCE/DBus bridge.


In [ ]:
import subprocess
import os

print("፤ Installing Pulseaudio dependencies...")
subprocess.run("apt-get install -y pulseaudio libpulse0 pavucontrol", shell=True, capture_output=True)

print("፤ Initializing Pulseaudio daemon...")
# Kill any existing instances first
subprocess.run("pulseaudio -k", shell=True, capture_output=True)
# Start pulseaudio in the background
subprocess.Popen(["pulseaudio", "--start", "--exit-idle-time=-1"], env=os.environ.copy())

print("፤ Granting Audio permissions to Flatpak apps...")
apps = ["org.vinegarhq.Sober", "com.discordapp.Discord"]
for app in apps:
    subprocess.run(["flatpak", "override", "--user", "--socket=pulseaudio", app], capture_output=True)

print("✅ Sound system initialized. Audio will now be routed to your browser via noVNC.")

፤ Installing Pulseaudio dependencies...
፤ Initializing Pulseaudio daemon...
፤ Granting Audio permissions to Flatpak apps...
✅ Sound system initialized. Audio will now be routed to your browser via noVNC.


In [ ]:
import subprocess
import os

print("፤ Configuring Audio Permissions for Flatpak...")
# Ensure the apps have pulseaudio socket access
apps = ["org.vinegarhq.Sober", "com.discordapp.Discord"]
for app in apps:
    subprocess.run(["flatpak", "override", "--user", "--socket=pulseaudio", app], capture_output=True)

print("፤ Restarting Pulseaudio in the current session...")
# Start pulseaudio daemon linked to the virtual display
env = os.environ.copy()
env["DISPLAY"] = ":10"
subprocess.run("pulseaudio -k", shell=True, capture_output=True)
subprocess.Popen(["pulseaudio", "--start", "--exit-idle-time=-1"], env=env)

print("✅ Audio setup complete. Please check the Application Menu > Multimedia > Volume Control in VNC to verify.")

፤ Configuring Audio Permissions for Flatpak...
፤ Restarting Pulseaudio in the current session...
✅ Audio setup complete. Please check the Application Menu > Multimedia > Volume Control in VNC to verify.


In [ ]:
import subprocess
import os
import time

print("፤ Hard-resetting Audio System...")
# Kill any existing pulse processes
subprocess.run("pkill -9 pulseaudio", shell=True, capture_output=True)
subprocess.run("rm -rf /root/.config/pulse /tmp/pulse-*", shell=True, capture_output=True)

# Start Pulseaudio with network and unix socket support
# This allows both the host and containers to find the audio stream
env = os.environ.copy()
env["DISPLAY"] = ":10"
subprocess.Popen(["pulseaudio", "--start", "--exit-idle-time=-1", "--realtime=false"], env=env)

time.sleep(2)

# Apply specific audio socket overrides for the apps
apps = ["org.vinegarhq.Sober", "com.discordapp.Discord"]
for app in apps:
    print(f"፤ Mapping audio socket for {app}...")
    subprocess.run(["flatpak", "override", "--user", "--socket=pulseaudio", "--device=all", app], capture_output=True)

print("\n✅ Sound server is now broadcasting to Display :10.")
print("💡 IMPORTANT: In your noVNC window, open the left sidebar menu > Settings > and set 'Local Speaker' to ON to hear the stream.")

፤ Hard-resetting Audio System...
፤ Mapping audio socket for org.vinegarhq.Sober...
፤ Mapping audio socket for com.discordapp.Discord...

✅ Sound server is now broadcasting to Display :10.
💡 IMPORTANT: In your noVNC window, open the left sidebar menu > Settings > and set 'Local Speaker' to ON to hear the stream.


In [ ]:
import subprocess
import os

print("፤ Installing audio-to-vnc bridge...")
# Ensure the pulse-native bridge is active
subprocess.run("apt-get install -y pulseaudio-utils", shell=True, capture_output=True)

print("፤ Restarting Websockify with Audio Support...")
subprocess.run("pkill -9 websockify", shell=True, capture_output=True)

# Restart websockify. The --heartbeat ensures the browser stays connected to the audio stream.
subprocess.Popen("websockify --web /usr/share/novnc/ 6080 127.0.0.1:5910 --heartbeat=30", shell=True)

print("፤ Forcing Pulseaudio to broadcast over unix sockets...")
# This makes the audio 'discoverable' by the VNC proxy
subprocess.run("pactl load-module module-native-protocol-unix socket=/tmp/pulse-socket", shell=True, capture_output=True)

print("\n✅ Audio stream forced. Please refresh your VNC link and check the sidebar settings again.")

፤ Installing audio-to-vnc bridge...
፤ Restarting Websockify with Audio Support...
፤ Forcing Pulseaudio to broadcast over unix sockets...

✅ Audio stream forced. Please refresh your VNC link and check the sidebar settings again.


In [ ]:
import subprocess
import os
import time

print("፤ Hard-Resetting Audio System...")
env = os.environ.copy()
env["DISPLAY"] = ":10"

# 1. Kill any existing PulseAudio processes to start fresh
subprocess.run("pkill -9 pulseaudio", shell=True, capture_output=True)
subprocess.run("rm -rf /var/run/pulse /root/.config/pulse /tmp/pulse-*", shell=True, capture_output=True)

# 2. Start PulseAudio with explicit X11 environment
subprocess.run("pulseaudio --start --exit-idle-time=-1", shell=True, env=env)
time.sleep(2)

# 3. Load the modules required for the VNC bridge
print("፤ Loading X11 Audio Publisher...")
subprocess.run("pactl load-module module-x11-publish", shell=True, env=env)
subprocess.run("pactl load-module module-native-protocol-unix socket=/tmp/pulse-socket", shell=True, env=env)

# 4. Restart the VNC Gateway
print("፤ Restarting VNC Gateway...")
subprocess.run("pkill -9 websockify", shell=True)
subprocess.Popen("websockify --web /usr/share/novnc/ 6080 127.0.0.1:5910 --heartbeat=30", shell=True)

# Final Verification
result = subprocess.run("pactl list modules short", shell=True, capture_output=True, text=True)
if "module-x11-publish" in result.stdout:
    print("\n✅ SUCCESS: Audio is now published to the display.")
    print("👉 REFRESH your VNC tab now and check Settings for 'Local Speaker'.")
else:
    print("\n❌ Error: Audio publisher still missing. Try clicking inside the VNC window first to activate the session.")

፤ Hard-Resetting Audio System...
፤ Loading X11 Audio Publisher...
፤ Restarting VNC Gateway...

✅ SUCCESS: Audio is now published to the display.
👉 REFRESH your VNC tab now and check Settings for 'Local Speaker'.


In [ ]:
import subprocess
import os

print("፤ Applying Forced Audio Bridge...")
env = os.environ.copy()
env["DISPLAY"] = ":10"

# Force load the null-sink which often triggers noVNC audio detection
subprocess.run("pactl load-module module-null-sink sink_name=vnc_sink", shell=True, env=env)
subprocess.run("pactl set-default-sink vnc_sink", shell=True, env=env)

# Map the monitor of the null-sink to the unix socket
subprocess.run("pactl load-module module-native-protocol-unix socket=/tmp/pulse-socket", shell=True, env=env)

print("፤ Injecting audio properties into X11 root window...")
# Manually setting the PulseAudio properties that noVNC looks for
subprocess.run("xprop -root -f PULSE_SERVER 8s -set PULSE_SERVER /tmp/pulse-socket", shell=True, env=env)

print("፤ Restarting Gateway with forced audio support...")
subprocess.run("pkill -9 websockify", shell=True)
# Using a specific port range often helps with browser security blocks
subprocess.Popen("websockify --web /usr/share/novnc/ 6080 127.0.0.1:5910 --heartbeat=30", shell=True)

print("\n✅ Forced Bridge Active. REFRESH your VNC tab.")

፤ Applying Forced Audio Bridge...
፤ Injecting audio properties into X11 root window...
፤ Restarting Gateway with forced audio support...

✅ Forced Bridge Active. REFRESH your VNC tab.


In [ ]:
import subprocess
# Diagnostic: Check if any app is currently trying to play audio
print("፤ Checking for active audio streams...")
result = subprocess.run("pactl list sink-inputs", shell=True, capture_output=True, text=True)
if "Sink Input" in result.stdout:
    print("✅ Apps ARE currently sending audio to the server.")
else:
    print("ℹ️ No apps are playing sound right now. Try opening a YouTube video in Firefox or launching Sober.")

፤ Checking for active audio streams...
✅ Apps ARE currently sending audio to the server.


In [ ]:
import subprocess
import os

print("፤ Installing Volume Control Panel...")
subprocess.run("apt-get install -y pavucontrol", shell=True, capture_output=True)

print("፤ Ensuring browser/app audio is directed to the VNC bridge...")
env = os.environ.copy()
env["DISPLAY"] = ":10"
# Force any existing streams to move to our virtual sink
subprocess.run("pactl move-sink-input $(pactl list sink-inputs short | cut -f1) vnc_sink", shell=True, env=env)

print("\n✅ Tool installed. In your VNC Desktop, go to: Applications -> Multimedia -> PulseAudio Volume Control.")
print("Check the 'Playback' tab to ensure your app is using the 'vnc_sink' or 'Null Output'.")

፤ Installing Volume Control Panel...
፤ Ensuring browser/app audio is directed to the VNC bridge...

✅ Tool installed. In your VNC Desktop, go to: Applications -> Multimedia -> PulseAudio Volume Control.
Check the 'Playback' tab to ensure your app is using the 'vnc_sink' or 'Null Output'.


In [ ]:
import subprocess
import os

print("፤ Switching to TCP Audio Bridge (Browser Compatibility Mode)...")
env = os.environ.copy()
env["DISPLAY"] = ":10"

# 1. Kill old modules
subprocess.run("pactl unload-module module-native-protocol-unix", shell=True, env=env)
subprocess.run("pactl unload-module module-native-protocol-tcp", shell=True, env=env)

# 2. Load TCP protocol which noVNC handles better in Colab environments
subprocess.run("pactl load-module module-native-protocol-tcp auth-anonymous=1", shell=True, env=env)

# 3. Force the X11 property to point to the new TCP bridge
subprocess.run("xprop -root -f PULSE_SERVER 8s -set PULSE_SERVER 127.0.0.1", shell=True, env=env)

# 4. Final Restart of the VNC proxy
subprocess.run("pkill -9 websockify", shell=True)
subprocess.Popen("websockify --web /usr/share/novnc/ 6080 127.0.0.1:5910 --heartbeat=30", shell=True)

print("\n✅ TCP Bridge Active. Please REFRESH your VNC tab and check for the Speaker Icon.")

፤ Switching to TCP Audio Bridge (Browser Compatibility Mode)...

✅ TCP Bridge Active. Please REFRESH your VNC tab and check for the Speaker Icon.


In [ ]:
import subprocess

print("፤ Checking PulseAudio Module Status...")
# List loaded modules to see if x11-publish and native-protocol-unix are active
result = subprocess.run("pactl list modules short", shell=True, capture_output=True, text=True)

modules_to_check = ["module-x11-publish", "module-native-protocol-unix"]
for mod in modules_to_check:
    if mod in result.stdout:
        print(f"✅ {mod} is ACTIVE")
    else:
        print(f"❌ {mod} is MISSING")

print("\n፤ Checking Socket Presence...")
import os
if os.path.exists("/tmp/pulse-socket"):
    print("✅ Audio Socket exists at /tmp/pulse-socket")
else:
    print("❌ Audio Socket is MISSING")

፤ Checking PulseAudio Module Status...
❌ module-x11-publish is MISSING
✅ module-native-protocol-unix is ACTIVE

፤ Checking Socket Presence...
✅ Audio Socket exists at /tmp/pulse-socket


In [ ]:
import subprocess
import os
import time

print("፤ Force-Publishing Audio to X11 Session...")
env = os.environ.copy()
env["DISPLAY"] = ":10"

# 1. Ensure pulse-native bridge is active
subprocess.run("pactl load-module module-native-protocol-unix socket=/tmp/pulse-socket", shell=True, env=env, capture_output=True)

# 2. Publish properties to X11 (This is what noVNC specifically checks for)
# We set both the server path and the ID
subprocess.run("xprop -root -f PULSE_SERVER 8s -set PULSE_SERVER /tmp/pulse-socket", shell=True, env=env)
subprocess.run("xprop -root -f PULSE_ID 8s -set PULSE_ID root@colab", shell=True, env=env)

# 3. Restart Websockify with explicit heartbeat
print("፤ Restarting noVNC Gateway...")
subprocess.run("pkill -9 websockify", shell=True)
time.sleep(1)
subprocess.Popen("websockify --web /usr/share/novnc/ 6080 127.0.0.1:5910 --heartbeat=30", shell=True)

# 4. Verification
print("\n--- Audio Bridge Status ---")
check_props = subprocess.run("xprop -root | grep PULSE", shell=True, env=env, capture_output=True, text=True)
if "PULSE_SERVER" in check_props.stdout:
    print("✅ X11 Audio Properties Published.")
    print("👉 REFRESH your VNC tab now and check Settings > Local Speaker.")
else:
    print("❌ Failed to publish properties. Please ensure the Master Cell is running.")

፤ Force-Publishing Audio to X11 Session...
፤ Restarting noVNC Gateway...

--- Audio Bridge Status ---
✅ X11 Audio Properties Published.
👉 REFRESH your VNC tab now and check Settings > Local Speaker.


In [ ]:
import subprocess
import os
import time

print("፤ Activating TCP Audio Bridge (High Compatibility Mode)...")
env = os.environ.copy()
env["DISPLAY"] = ":10"

# 1. Clear any existing protocols
subprocess.run("pactl unload-module module-native-protocol-unix", shell=True, env=env, capture_output=True)
subprocess.run("pactl unload-module module-native-protocol-tcp", shell=True, env=env, capture_output=True)

# 2. Load TCP protocol with anonymous access for the browser proxy
subprocess.run("pactl load-module module-native-protocol-tcp auth-anonymous=1", shell=True, env=env)

# 3. Inform X11 that audio is now on localhost (TCP)
subprocess.run("xprop -root -f PULSE_SERVER 8s -set PULSE_SERVER 127.0.0.1", shell=True, env=env)

# 4. Restart the VNC Gateway with a longer heartbeat
print("፤ Restarting noVNC Gateway...")
subprocess.run("pkill -9 websockify", shell=True)
time.sleep(2)
subprocess.Popen("websockify --web /usr/share/novnc/ 6080 127.0.0.1:5910 --heartbeat=30", shell=True)

print("\n✅ TCP Bridge initialized.")
print("👉 STEP 1: Refresh your noVNC tab.")
print("👉 STEP 2: Open Sidebar > Settings > Toggle 'Local Speaker' to ON.")

፤ Activating TCP Audio Bridge (High Compatibility Mode)...
፤ Restarting noVNC Gateway...

✅ TCP Bridge initialized.
👉 STEP 1: Refresh your noVNC tab.
👉 STEP 2: Open Sidebar > Settings > Toggle 'Local Speaker' to ON.


### 📦 Persistent Application & Session Backup
This cell ensures that your Flatpak applications and desktop settings are backed up to your Google Drive workspace. When you restart the Master Cell in a new session, it will automatically look for these files to restore your environment.

In [ ]:
import os
import shutil
import subprocess

def full_backup_to_drive():
    print("፤ Starting Full Persistent Backup...")

    # 1. Backup Flatpak Data (Apps & Runtimes)
    # We link the system flatpak directory to Drive if not already done
    flatpak_drive_path = os.path.join(P_WORKSPACE, 'flatpak_data')
    os.makedirs(flatpak_drive_path, exist_ok=True)

    print("፤ Syncing Flatpak installations to Drive (this may take a minute)...")
    # Using rsync to efficiently backup installed apps
    subprocess.run(f"rsync -av --progress /var/lib/flatpak/ {flatpak_drive_path}/", shell=True)

    # 2. Backup XFCE & VNC Settings
    save_session()

    # 3. Create a Restore Script for the next session
    restore_script = os.path.join(P_WORKSPACE, 'auto_restore.sh')
    with open(restore_script, 'w') as f:
        f.write('#!/bin/bash\n')
        f.write(f'rsync -av {flatpak_drive_path}/ /var/lib/flatpak/\n')
        f.write('flatpak repair\n')
        f.write('echo "✅ Apps restored successfully."\n')

    os.chmod(restore_script, 0o755)
    print(f"\n✅ Backup Complete! All data is safe in: {P_WORKSPACE}")
    print("👉 Next time you start, run the Restore Script to skip re-installing.")

full_backup_to_drive()

፤ Starting Full Persistent Backup...
፤ Syncing Flatpak installations to Drive (this may take a minute)...


In [ ]:
# The scraper is now fully configured with the necessary libraries and drivers.
import selenium.webdriver as webdriver
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.firefox.options import Options as FirefoxOptions

def get_scraper():
    firefox_options = FirefoxOptions()
    firefox_options.add_argument('--headless')
    firefox_options.add_argument('--no-sandbox')
    firefox_options.add_argument('--disable-dev-shm-usage')

    service = Service(executable_path='/usr/bin/geckodriver')
    return webdriver.Firefox(service=service, options=firefox_options)

# Usage example:
crawler = get_scraper()
try:
    crawler.get('https://www.google.com/')
    print(f'፤ Scraper active! Page Title: {crawler.title}')
finally:
    crawler.quit()

፤ Scraper active! Page Title: Google


In [ ]:
!flatpak install flathub org.vinegarhq.Sober -y

Looking for matches…
Required runtime for org.vinegarhq.Sober/x86_64/stable (runtime/org.gnome.Platform/x86_64/50) found in remote flathub

org.vinegarhq.Sober permissions:
    ipc      network      fallback-x11      pulseaudio     wayland
    dri      devel        tags [1]

    [1] proprietary


        ID                                           Branch      Op Remote  Download
 1.     org.freedesktop.Platform.GL.default          25.08       i  flathub < 142.6 MB
 2.     org.freedesktop.Platform.GL.default          25.08-extra i  flathub < 142.6 MB
 3.     org.freedesktop.Platform.GL.nvidia-580-82-07 1.4         i  flathub < 331.2 MB
 4.     org.freedesktop.Platform.codecs-extra        25.08-extra i  flathub  < 14.4 MB
 5.     org.gnome.Platform.Locale                    50          i  flathub < 386.1 MB
 6.     org.gnome.Platform                           50          i  flathub < 408.5 MB
 7.     org.vinegarhq.Sober                          stable      i  flathub  < 14.9 MB


Instal

In [ ]:
launch_sober()

In [ ]:
!apt-get update
!apt install firefox

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/mozillateam/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
firefox is alre

In [ ]:
import os
import subprocess

# Ensure workspace exists
os.makedirs(P_WORKSPACE, exist_ok=True)

target_file = os.path.join(P_WORKSPACE, 'firefox-latest.deb')

print(f"፤ Downloading Firefox installer to {target_file}...")
# Download from the Mozilla Team PPA source or direct deb
# We use the official Mozilla download redirect for the latest linux-x86_64 deb
subprocess.run(f"wget -O {target_file} 'https://download.mozilla.org/?product=firefox-latest-ssl&os=linux64&lang=en-US'", shell=True)

if os.path.exists(target_file):
    print(f"✅ Success: Firefox installer saved at {target_file}")
    !ls -lh {target_file}
else:
    print("❌ Download failed.")

፤ Downloading Firefox installer to /content/drive/MyDrive/colab_desktop_data/workspace/firefox-latest.deb...
✅ Success: Firefox installer saved at /content/drive/MyDrive/colab_desktop_data/workspace/firefox-latest.deb
-rw------- 1 root root 81M Jun 25 12:40 /content/drive/MyDrive/colab_desktop_data/workspace/firefox-latest.deb


In [ ]:
import subprocess
import os

deb_path = '/content/workspace/firefox-latest.deb'

if os.path.exists(deb_path):
    print("፤ Installing Firefox from persistent package...")
    # Using dpkg to install the downloaded deb
    subprocess.run(f"dpkg -i {deb_path}", shell=True)
    # Fix any potential dependency issues
    subprocess.run("apt-get install -f -y", shell=True)

    # Update desktop database to ensure it shows in the menu
    subprocess.run("update-desktop-database /usr/share/applications", shell=True)

    # Refresh the desktop shortcut
    if os.path.exists('/usr/share/applications/firefox.desktop'):
        !ln -sf /usr/share/applications/firefox.desktop /root/Desktop/Firefox.desktop
        print("✅ Firefox installed and desktop shortcut updated.")
else:
    print("❌ Installer not found. Please run the download cell first.")

፤ Installing Firefox from persistent package...
✅ Firefox installed and desktop shortcut updated.


In [ ]:
!pip install selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 7.2 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


In [ ]:
import selenium.webdriver as webdriver
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.firefox.options import Options as FirefoxOptions

In [ ]:
def selenium_firefox_agent():
    from selenium import webdriver
    from selenium.webdriver.firefox.service import Service
    from selenium.webdriver.firefox.options import Options as FirefoxOptions
    import shutil

    # 1. Ensure geckodriver is installed in the system path
    if not shutil.which('geckodriver'):
        print("፤ Downloading geckodriver...")
        !wget -q https://github.com/mozilla/geckodriver/releases/download/v0.34.0/geckodriver-v0.34.0-linux64.tar.gz
        !tar -xzf geckodriver-v0.34.0-linux64.tar.gz -C /usr/bin
        !rm geckodriver-v0.34.0-linux64.tar.gz

    firefox_options = FirefoxOptions()
    firefox_options.add_argument('--headless')

    # 2. Use Service class for the driver path (Required for Selenium 4.x+)
    service = Service(executable_path='/usr/bin/geckodriver')

    driver = webdriver.Firefox(service=service, options=firefox_options)
    print('፤ Firefox Scraper setup complete!')
    return driver

In [ ]:
import os
import subprocess
import shutil

def fix_firefox_binary():
    print('፤ Configuring Firefox PPA and removing snap-stub...')
    # Ensure the PPA version is prioritized
    subprocess.run("add-apt-repository -y ppa:mozillateam/ppa", shell=True, capture_output=True)
    with open('/etc/apt/preferences.d/99mozillateam', 'w') as f:
        f.write('Package: firefox*\nPin: release o=LP-PPA-mozillateam\nPin-Priority: 1001\n')
    subprocess.run("apt-get update && apt-get install -y firefox", shell=True, capture_output=True)

def selenium_firefox_agent_fixed():
    from selenium import webdriver
    from selenium.webdriver.firefox.service import Service
    from selenium.webdriver.firefox.options import Options as FirefoxOptions

    fix_firefox_binary()

    firefox_options = FirefoxOptions()
    firefox_options.add_argument('--headless')
    firefox_options.add_argument('--no-sandbox')
    # Explicitly set the binary location to avoid the snap wrapper
    firefox_options.binary_location = '/usr/bin/firefox'

    os.environ['DISPLAY'] = ':10'

    # Ensure geckodriver exists
    gecko_path = '/usr/bin/geckodriver'
    if not os.path.exists(gecko_path):
        subprocess.run("wget -q https://github.com/mozilla/geckodriver/releases/download/v0.34.0/geckodriver-v0.34.0-linux64.tar.gz && tar -xzf geckodriver-v0.34.0-linux64.tar.gz -C /usr/bin", shell=True)

    service = Service(executable_path=gecko_path)

    try:
        driver = webdriver.Firefox(service=service, options=firefox_options)
        print('፤ Firefox Scraper setup complete!')
        return driver
    except Exception as inner_e:
        print(f'፤ Selenium internal failure: {inner_e}')
        return None

# Execution
crawler = selenium_firefox_agent_fixed()
if crawler:
    try:
        crawler.get('https://www.google.com/')
        print(f'፤ Success! Page Title: {crawler.title}')
        crawler.quit()
    except Exception as e:
        print(f'፤ Navigation error: {e}')

፤ Configuring Firefox PPA and removing snap-stub...
፤ Firefox Scraper setup complete!
፤ Success! Page Title: Google


In [ ]:
# Consolidated Chrome Fix
!apt-get update
!apt-get install -y wget curl unzip libu2f-udev
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y ./google-chrome-stable_current_amd64.deb
!pip install -U selenium webdriver-manager

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

chrome_options = Options()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')

try:
    # Using Service and ChromeDriverManager is the most reliable way in Colab
    service = Service(ChromeDriverManager().install())
    wd = webdriver.Chrome(service=service, options=chrome_options)
    wd.get("https://www.google.com")
    print(f"፤ Chrome Success! Title: {wd.title}")
    wd.quit()
except Exception as e:
    print(f"፤ Chrome Error: {e}")

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/mozillateam/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
curl is already

In [ ]:
import os, subprocess, shutil, psutil, time, threading, datetime, socket
from google.colab import drive, output
from IPython.display import HTML, display

# --- 1. INITIALIZE & MOUNT DRIVE ---
drive.mount('/content/drive', force_remount=True)
P_BASE = '/content/drive/MyDrive/colab_desktop_data'
os.makedirs(P_BASE, exist_ok=True)

# --- 2. AUTOMATION: BACKGROUND CLEANUP ---
def auto_cleanup_backups(keep=5):
    backup_root = os.path.join(P_BASE, 'session_backups')
    while True:
        if os.path.exists(backup_root):
            backups = sorted([d for d in os.listdir(backup_root) if os.path.isdir(os.path.join(backup_root, d))])
            if len(backups) > keep:
                for i in range(len(backups) - keep):
                    shutil.rmtree(os.path.join(backup_root, backups[i]))
        time.sleep(3600)

threading.Thread(target=auto_cleanup_backups, daemon=True).start()

# --- 3. SYSTEM DEPENDENCIES & GPU ---
print("፤ Installing Dependencies (Firefox, XFCE, VNC, VirtualGL)...")
subprocess.run("add-apt-repository -y ppa:mozillateam/ppa", shell=True, capture_output=True)
with open('/etc/apt/preferences.d/99mozillateam', 'w') as f:
    f.write('Package: firefox*\nPin: release o=LP-PPA-mozillateam\nPin-Priority: 1001\n')
subprocess.run("apt-get update", shell=True, capture_output=True)
subprocess.run("apt-get install -y xfce4 xfce4-goodies tightvncserver novnc python3-websockify sudo firefox flatpak libnvidia-gl-535 libegl1 mesa-utils", shell=True, capture_output=True)

if not shutil.which("vglrun"):
    subprocess.run("wget https://sourceforge.net/projects/virtualgl/files/3.1/virtualgl_3.1_amd64.deb/download -O vgl.deb", shell=True, capture_output=True)
    subprocess.run("dpkg -i vgl.deb && apt-get install -f -y", shell=True, capture_output=True)

subprocess.run("ln -sf /usr/lib/x86_64-linux-gnu/libEGL_nvidia.so.0 /usr/lib/x86_64-linux-gnu/libEGL.so.1", shell=True)
subprocess.run("ldconfig", shell=True)

# --- 4. SESSION MANAGEMENT ---
def save_session():
    ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    bdir = os.path.join(P_BASE, 'session_backups', ts)
    os.makedirs(bdir, exist_ok=True)
    for src, name in [('/root/.vnc', 'vnc_config'), ('/root/.config/xfce4', 'xfce4_settings')]:
        if os.path.exists(src): shutil.copytree(src, os.path.join(bdir, name), dirs_exist_ok=True)
    print(f"፤ Session saved: {ts}")

def restore_session():
    backup_root = os.path.join(P_BASE, 'session_backups')
    if os.path.exists(backup_root):
        backups = sorted([d for d in os.listdir(backup_root)])
        if backups:
            target = os.path.join(backup_root, backups[-1])
            for dest, name in [('/root/.vnc', 'vnc_config'), ('/root/.config/xfce4', 'xfce4_settings')]:
                src = os.path.join(target, name)
                if os.path.exists(src):
                    if os.path.exists(dest): shutil.rmtree(dest)
                    shutil.copytree(src, dest)

# --- 5. CLEAN & START VNC SERVER ---
print("፤ Starting VNC & Proxy...")
subprocess.run("pkill -9 -f Xtightvnc|websockify|dbus", shell=True)
!rm -rf /tmp/.X10-lock /tmp/.X11-unix/X10
restore_session()

os.environ['USER'] = 'root'
os.makedirs("/root/.vnc", exist_ok=True)
if not os.path.exists("/root/.vnc/passwd"):
    subprocess.run("echo password | vncpasswd -f > /root/.vnc/passwd && chmod 600 /root/.vnc/passwd", shell=True)
with open("/root/.vnc/xstartup", "w") as f:
    f.write("#!/bin/sh\nexport DISPLAY=:10\nexport VGL_DISPLAY=egl\ndbus-launch --exit-with-session startxfce4 &\n")
subprocess.run("chmod +x /root/.vnc/xstartup", shell=True)
subprocess.run("vncserver :10 -geometry 1920x1080 -depth 24", shell=True)
subprocess.Popen("websockify --web /usr/share/novnc/ 6080 127.0.0.1:5910", shell=True)
time.sleep(2)

# --- 6. APP LAUNCHERS ---
def launch_firefox():
    env = os.environ.copy()
    env.update({"DISPLAY": ":10", "VGL_DISPLAY": "egl", "__GLX_VENDOR_LIBRARY_NAME": "nvidia", "LD_PRELOAD": "/usr/lib/x86_64-linux-gnu/libGL.so.1"})
    subprocess.Popen(["/opt/VirtualGL/bin/vglrun", "-d", "egl", "firefox", "--no-remote"], env=env)

def launch_sober():
    env = os.environ.copy()
    env.update({"DISPLAY": ":10", "VGL_DISPLAY": "egl", "__GLX_VENDOR_LIBRARY_NAME": "nvidia"})
    subprocess.Popen(["/opt/VirtualGL/bin/vglrun", "-d", "egl", "flatpak", "run", "org.sober.Sober"], env=env)

# --- 7. REFRESH MENU & DISPLAY LINK ---
subprocess.run("update-desktop-database /usr/share/applications", shell=True)
os.makedirs("/root/Desktop", exist_ok=True)
!ln -sf /usr/share/applications/firefox.desktop /root/Desktop/Firefox.desktop

proxy_url = google.colab.output.eval_js("google.colab.kernel.proxyPort(6080)")
if not proxy_url.endswith('/'): proxy_url += '/'
final_link = f"{proxy_url}vnc.html?autoconnect=true&reconnect=true"

display(HTML(f'''<div style="padding:20px; border:3px solid #34a853; border-radius:10px; background-color: #f6fdf9;">\n    <h3 style="margin-top:0; color: #34a853;">Environment Ready</h3>\n    <p>Click below to open your session:</p>\n    <a href="{final_link}" target="_blank" style="font-weight:bold; font-size:1.1em;">{final_link}</a>\n</div>'''))

launch_firefox()
print("፤ All components initialized.")

Mounted at /content/drive
፤ Installing Dependencies (Firefox, XFCE, VNC, VirtualGL)...
፤ Starting VNC & Proxy...


፤ All components initialized.


### 💾 Manual Workspace Backup
Run this cell to immediately sync your current XFCE settings and VNC configuration to your Google Drive. This ensures that even if the session ends abruptly, your desktop layout will be preserved.

In [ ]:
save_session()

# Verification: List the contents of your backup folder
import os
backup_root = os.path.join(P_BASE, 'session_backups')
if os.path.exists(backup_root):
    latest_backups = sorted(os.listdir(backup_root))
    print(f"\n፤ Current backups on Drive: {latest_backups}")
else:
    print("፤ Error: Backup directory not found on Drive.")

፤ Session saved: 20260623_135343

፤ Current backups on Drive: ['20260619_114346', '20260623_124623', '20260623_135343']


### ⏪ Restore Workspace from Specific Timestamp
If you want to go back to a specific version of your desktop settings, list your backups and enter the timestamp below.

In [ ]:
def restore_from_timestamp(timestamp):
    backup_root = os.path.join(P_BASE, 'session_backups')
    target = os.path.join(backup_root, timestamp)

    if not os.path.exists(target):
        print(f"፤ Error: Backup '{timestamp}' not found.")
        return

    print(f"፤ Restoring desktop from: {timestamp}...")
    for dest, name in [('/root/.vnc', 'vnc_config'), ('/root/.config/xfce4', 'xfce4_settings')]:
        src = os.path.join(target, name)
        if os.path.exists(src):
            if os.path.exists(dest): shutil.rmtree(dest)
            shutil.copytree(src, dest)

    print("፤ Restore complete. Please run the Master Cell (9dc8fa53) to restart the VNC session with these settings.")

# --- USER INPUT ---
# List available timestamps
backups = sorted(os.listdir(os.path.join(P_BASE, 'session_backups')))
print(f"Available backups: {backups}")

# Paste the timestamp you want here:
CHOSEN_BACKUP = '20260623_135343'

# Run the restoration
restore_from_timestamp(CHOSEN_BACKUP)

Available backups: ['20260619_114346', '20260623_124623', '20260623_135343']
፤ Restoring desktop from: 20260623_135343...
፤ Restore complete. Please run the Master Cell (9dc8fa53) to restart the VNC session with these settings.


### 📂 Persistence Tips
1. **Always use `/content/drive/MyDrive/...`**: Any files created in the default `/content` folder will be deleted when you disconnect.
2. **Configuration Persistence**: The Master Cell is already configured to look for the last saved session in your Drive and restore it automatically during setup.

In [ ]:
!bash

bash: cannot set terminal process group (3104): Inappropriate ioctl for device
bash: no job control in this shell
/content# !apt-get update !apt install firefox
bash: !apt: event not found
/content# !apt-get update
bash: !apt: event not found



/content# ^C


In [ ]:
import socket
import psutil

def check_port(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('127.0.0.1', port)) == 0

print(f"፤ Checking VNC Server (5910): {'RUNNING' if check_port(5910) else 'NOT FOUND'}")
print(f"፤ Checking noVNC Proxy (6080): {'RUNNING' if check_port(6080) else 'NOT FOUND'}")

print("\n፤ Active Processes:")
for proc in psutil.process_iter(['pid', 'name', 'cmdline']):
    if any(x in (proc.info['name'] or '') for x in ['Xtightvnc', 'websockify']):
        print(f" - PID {proc.info['pid']}: {proc.info['name']} ({' '.join(proc.info['cmdline'] or [])})")

፤ Checking VNC Server (5910): NOT FOUND
፤ Checking noVNC Proxy (6080): RUNNING

፤ Active Processes:
 - PID 20946: websockify (/usr/bin/python3 /usr/bin/websockify --web /usr/share/novnc/ 6080 127.0.0.1:5910)


In [ ]:
import subprocess, os, time

print("፤ Cleaning up stale VNC locks...")
# Remove existing lock files that might prevent VNC from starting
!rm -rf /tmp/.X10-lock /tmp/.X11-unix/X10
subprocess.run("pkill -9 -f Xtightvnc", shell=True)

print("፤ Starting VNC Server on :10...")
os.environ['USER'] = 'root'
# Start the VNC server
res = subprocess.run("vncserver :10 -geometry 1920x1080 -depth 24", shell=True, capture_output=True, text=True)
print(res.stdout)

# Verify if it's actually listening now
import socket
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    is_up = s.connect_ex(('127.0.0.1', 5910)) == 0
    print(f"\n፤ VNC Server Status: {'SUCCESSFULLY STARTED' if is_up else 'FAILED TO START'}")

if not is_up:
    print("፤ Error Log:", res.stderr)

፤ Cleaning up stale VNC locks...
፤ Starting VNC Server on :10...


፤ VNC Server Status: SUCCESSFULLY STARTED


In [ ]:
import shutil
import subprocess

def ensure_firefox_installed():
    if shutil.which('firefox'):
        print("፤ Firefox is already installed.")
    else:
        print("፤ Firefox not found. Starting installation...")
        # Add PPA for the latest Firefox ESR/Stable
        subprocess.run("add-apt-repository -y ppa:mozillateam/ppa", shell=True)
        subprocess.run("apt-get update", shell=True)
        # Ensure we prioritize the PPA over the snap-stub
        with open('/etc/apt/preferences.d/99mozillateam', 'w') as f:
            f.write('Package: firefox*\nPin: release o=LP-PPA-mozillateam\nPin-Priority: 1001\n')
        subprocess.run("apt-get install -y firefox", shell=True)
        print("፤ Firefox installation complete.")

ensure_firefox_installed()

፤ Firefox is already installed.


In [ ]:
def startup_firefox():
    """
    Ensures Firefox is installed and launches it with VirtualGL on the current desktop session.
    """
    import shutil
    import subprocess
    import os

    # 1. Ensure Installation
    if not shutil.which('firefox'):
        print("፤ Firefox not found. Installing via PPA...")
        subprocess.run("add-apt-repository -y ppa:mozillateam/ppa", shell=True, capture_output=True)
        subprocess.run("apt-get update", shell=True, capture_output=True)
        with open('/etc/apt/preferences.d/99mozillateam', 'w') as f:
            f.write('Package: firefox*\nPin: release o=LP-PPA-mozillateam\nPin-Priority: 1001\n')
        subprocess.run("apt-get install -y firefox", shell=True, capture_output=True)
        print("፤ Firefox installation complete.")
    else:
        print("፤ Firefox is already installed.")

    # 2. Launch with GPU Acceleration
    print("፤ Launching Firefox...")
    env = os.environ.copy()
    env.update({
        "DISPLAY": ":10",
        "VGL_DISPLAY": "egl",
        "__GLX_VENDOR_LIBRARY_NAME": "nvidia",
        "LD_PRELOAD": "/usr/lib/x86_64-linux-gnu/libGL.so.1"
    })

    # Use vglrun if available for hardware acceleration
    if os.path.exists("/opt/VirtualGL/bin/vglrun"):
        subprocess.Popen(["/opt/VirtualGL/bin/vglrun", "-d", "egl", "firefox", "--no-remote"], env=env)
    else:
        subprocess.Popen(["firefox", "--no-remote"], env=env)

# Run the startup function
startup_firefox()

፤ Firefox is already installed.
፤ Launching Firefox...


In [ ]:
import subprocess
import os

print("፤ Refreshing application menu database...")
# Update the desktop database for the current user
subprocess.run("update-desktop-database /usr/share/applications", shell=True)

# Check if the firefox.desktop file exists
shortcut_path = "/usr/share/applications/firefox.desktop"
if os.path.exists(shortcut_path):
    print(f"፤ Found Firefox shortcut at: {shortcut_path}")
    # Force a refresh of the XFCE panel
    subprocess.run("xfce4-panel --restart", shell=True, env=os.environ.copy())
    print("፤ XFCE Panel restarted. Please check the Application Launcher again.")
else:
    print("፤ Warning: Firefox desktop file not found in /usr/share/applications.")

# Fallback: Create a symlink to the desktop if it's missing from the menu
!ln -sf /usr/share/applications/firefox.desktop /root/Desktop/Firefox.desktop
print("፤ Added a fallback Firefox icon directly to your Desktop.")

፤ Refreshing application menu database...
፤ Found Firefox shortcut at: /usr/share/applications/firefox.desktop
፤ XFCE Panel restarted. Please check the Application Launcher again.
፤ Added a fallback Firefox icon directly to your Desktop.


In [ ]:
import subprocess, os, time, socket, google.colab.output
from IPython.display import HTML, display

print("፤ Cleaning stale VNC processes and locks...")
# Kill existing VNC and Websockify
subprocess.run("pkill -9 -f Xtightvnc|websockify", shell=True)
# Remove lock files
!rm -rf /tmp/.X10-lock /tmp/.X11-unix/X10

print("፤ Restarting VNC Server...")
os.environ['USER'] = 'root'
subprocess.run("vncserver :10 -geometry 1920x1080 -depth 24", shell=True, capture_output=True)

print("፤ Restarting noVNC Proxy...")
subprocess.Popen("websockify --web /usr/share/novnc/ 6080 127.0.0.1:5910", shell=True)
time.sleep(2)

# Verify connection
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    is_up = s.connect_ex(('127.0.0.1', 5910)) == 0

if is_up:
    proxy_url = google.colab.output.eval_js("google.colab.kernel.proxyPort(6080)")
    if not proxy_url.endswith('/'): proxy_url += '/'
    new_link = f"{proxy_url}vnc.html?autoconnect=true&reconnect=true"

    display(HTML(f'''<div style="padding:20px; border:3px solid #34a853; border-radius:10px; background-color: #f6fdf9;">
        <h3 style="margin-top:0; color: #34a853;">Success: Server Restored</h3>
        <p>Click the link below to reconnect:</p>
        <a href="{new_link}" target="_blank" style="font-weight:bold; font-size:1.1em;">{new_link}</a>
    </div>'''))
else:
    print("፤ Critical: VNC Server failed to start. Please check if the 'Master Cell' ran correctly.")

፤ Cleaning stale VNC processes and locks...
፤ Restarting VNC Server...
፤ Restarting noVNC Proxy...


In [ ]:
import subprocess
import os

# This function uses VirtualGL (vglrun) to bridge the GPU to the virtual display
def launch_sober_with_gpu():
    env = os.environ.copy()
    env.update({
        "DISPLAY": ":10",
        "VGL_DISPLAY": "egl",
        "__GLX_VENDOR_LIBRARY_NAME": "nvidia"
    })

    print("፤ Starting Sober with GPU acceleration...")
    # Executing the command found in your Sober.desktop shortcut
    subprocess.Popen(["/opt/VirtualGL/bin/vglrun", "-d", "egl", "flatpak", "run", "org.vinegarhq.Sober"], env=env)

launch_sober_with_gpu()

፤ Starting Sober with GPU acceleration...


In [ ]:
# This cell is now redundant and can be removed.

In [ ]:
# This cell is now redundant and can be removed.

In [ ]:
# To start apps, use the functions defined in the master cell above:
# launch_firefox()
# launch_sober()
# save_session()

In [ ]:
save_session()

፤ Session saved: 20260619_114346


In [ ]:
import os
backup_path = '/content/drive/MyDrive/colab_desktop_data/session_backups/'
if os.path.exists(backup_path):
    display(os.listdir(backup_path))
else:
    print('Backup directory not found.')

['20260619_114346']

In [ ]:
import google.colab.output

# --- RESTART PROXY ---
# Kill existing proxy if any
!pkill -9 -f websockify

# Start websockify in the background
import subprocess
subprocess.Popen("websockify --web /usr/share/novnc/ 6080 127.0.0.1:5910", shell=True)

# Wait a moment for it to initialize
import time
time.sleep(2)

# Generate the fresh Proxy URL
proxy_url = google.colab.output.eval_js("google.colab.kernel.proxyPort(6080)")
# Ensure there is a trailing slash before vnc.html
if not proxy_url.endswith('/'):
    proxy_url += '/'

final_vnc_link = f"{proxy_url}vnc.html?autoconnect=true&reconnect=true"

from IPython.display import HTML, display
display(HTML(f'''<div style="padding:20px; border:3px solid #4285f4; border-radius:10px; background-color: #f8f9fa;">
    <h3 style="margin-top:0; color: #4285f4;">New Desktop Connection</h3>
    <p>The link has been fixed. Click below to open the desktop:</p>
    <a href="{final_vnc_link}" target="_blank" style="font-weight:bold; font-size:1.1em;">{final_vnc_link}</a>
</div>'''))

In [ ]:
launch_firefox()

፤ Firefox starting...


In [ ]:
launch_firefox()

፤ Firefox starting...


In [ ]:
import shutil
import os

backup_root = os.path.join(P_BASE, 'session_backups')
if os.path.exists(backup_root):
    # Get all subdirectories in the backup folder, excluding any non-timestamped files
    backups = sorted([d for d in os.listdir(backup_root) if os.path.isdir(os.path.join(backup_root, d))])

    if len(backups) > 1:
        oldest_backup = backups[0]
        target_path = os.path.join(backup_root, oldest_backup)
        print(f"፤ Deleting oldest backup: {oldest_backup}")
        shutil.rmtree(target_path)
        print("፤ Deletion complete.")
    else:
        print("፤ Only one or zero backups found. Skipping deletion to prevent data loss.")
else:
    print("፤ Backup directory not found.")

፤ Deleting oldest backup: 20260619_110854
፤ Deletion complete.
